In [ ]:
import copy
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

PROJECT_ROOT = Path.cwd().parent
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed"
DESPECKLED_DIR = PROJECT_ROOT / "data" / "processed" / "despeckled"
MODEL_DIR = PROJECT_ROOT / "models"
TRAIN_RESULTS_DIR = PROJECT_ROOT / "results" / "training"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_ROOTS = {
    "original": PREPROCESSED_DIR,
    "gaussian": DESPECKLED_DIR / "gaussian",
    "median": DESPECKLED_DIR / "median",
    "bilateral": DESPECKLED_DIR / "bilateral",
    "non_local_means": DESPECKLED_DIR / "non_local_means",
}

# Fixed configuration (identical to notebook 03)
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 1e-4
THRESHOLD = 0.5

print("Device:", DEVICE)


# ------------------------------------------------------------
# Dataset (same preprocessing as notebook 03)
# ------------------------------------------------------------
class BUSIDataset(Dataset):
    def __init__(self, root_dir, split):
        root_dir = Path(root_dir)
        self.image_files = sorted((root_dir / split / "images").glob("*.png"))
        self.mask_files = sorted((root_dir / split / "masks").glob("*.png"))
        if [f.name for f in self.image_files] != [f.name for f in self.mask_files]:
            raise ValueError(f"Image/mask filenames do not match in {root_dir / split}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_files[idx]).convert("L"), dtype=np.float32) / 255.0
        mask = np.array(Image.open(self.mask_files[idx]).convert("L"), dtype=np.float32)
        mask = (mask > 0).astype(np.float32)
        return torch.from_numpy(image).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)


# ------------------------------------------------------------
# U-Net (exact architecture of the baseline checkpoint)
# ------------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.bottleneck = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        self.output = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.output(d1)


# ------------------------------------------------------------
# Loss and metrics (identical to notebook 03)
# ------------------------------------------------------------
def dice_loss(logits, targets, smooth=1e-6):
    probabilities = torch.sigmoid(logits)
    probabilities = probabilities.contiguous().view(probabilities.shape[0], -1)
    targets = targets.contiguous().view(targets.shape[0], -1)
    intersection = (probabilities * targets).sum(dim=1)
    denominator = probabilities.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1.0 - dice.mean()


bce_loss_fn = nn.BCEWithLogitsLoss()


def combined_loss(logits, targets):
    return bce_loss_fn(logits, targets) + dice_loss(logits, targets)


def calculate_metrics(logits, targets, threshold=0.5):
    predictions = (torch.sigmoid(logits) > threshold).float()
    predictions = predictions.view(predictions.shape[0], -1)
    targets = targets.view(targets.shape[0], -1)

    intersection = (predictions * targets).sum(dim=1)
    pred_sum = predictions.sum(dim=1)
    target_sum = targets.sum(dim=1)
    union = pred_sum + target_sum - intersection

    dice = (2 * intersection + 1e-6) / (pred_sum + target_sum + 1e-6)
    iou = (intersection + 1e-6) / (union + 1e-6)
    correct = (predictions == targets).float().sum(dim=1)
    pixel_accuracy = correct / targets.shape[1]

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "pixel_accuracy": pixel_accuracy.mean().item(),
    }


# ------------------------------------------------------------
# Train / validate (same procedure as notebook 03, plus diagnostics)
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss, total_samples = 0.0, 0
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = combined_loss(model(images), masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        total_samples += images.size(0)
    return running_loss / total_samples


def validate_one_epoch(model, loader):
    model.eval()
    running_loss, total_samples = 0.0, 0
    all_dice, all_iou, all_accuracy = [], [], []
    lesion_dice, normal_total, normal_fp = [], 0, 0

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            loss = combined_loss(logits, masks)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            metrics = calculate_metrics(logits, masks)
            all_dice.append(metrics["dice"])
            all_iou.append(metrics["iou"])
            all_accuracy.append(metrics["pixel_accuracy"])

            # Diagnostics only: never used for checkpointing or the scheduler
            preds = (torch.sigmoid(logits) > THRESHOLD).float().view(batch_size, -1)
            tgts = masks.view(batch_size, -1)
            inter = (preds * tgts).sum(dim=1)
            pred_sum, tgt_sum = preds.sum(dim=1), tgts.sum(dim=1)
            per_image = (2 * inter + 1e-6) / (pred_sum + tgt_sum + 1e-6)
            for d, p, t in zip(per_image.tolist(), pred_sum.tolist(), tgt_sum.tolist()):
                if t > 0:
                    lesion_dice.append(d)
                else:
                    normal_total += 1
                    normal_fp += int(p > 0)

    return {
        "loss": running_loss / total_samples,
        "dice": float(np.mean(all_dice)),
        "iou": float(np.mean(all_iou)),
        "pixel_accuracy": float(np.mean(all_accuracy)),
        "lesion_dice": float(np.mean(lesion_dice)),
        "normal_fp": normal_fp,
        "normal_total": normal_total,
    }


# ------------------------------------------------------------
# One complete training run
# ------------------------------------------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_experiment(condition, seed, epochs=EPOCHS, run_name=None):
    run_name = run_name or f"{condition}_seed{seed}"
    root = CONDITION_ROOTS[condition]

    train_ds = BUSIDataset(root, "train")
    val_ds = BUSIDataset(root, "validation")

    # Dedicated generator: same shuffle order for every condition at a given seed
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=shuffle_generator,
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    set_seed(seed)  # immediately before model creation -> same initial weights per seed
    model = UNet(in_channels=1, out_channels=1).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4
    )

    checkpoint_path = MODEL_DIR / f"{run_name}_best.pth"
    history_path = TRAIN_RESULTS_DIR / f"{run_name}_history.csv"

    history = []
    best_val_dice, best_epoch = -1.0, 0
    run_start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer)
        val = validate_one_epoch(model, val_loader)
        scheduler.step(val["dice"])
        current_lr = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val["loss"],
            "val_dice": val["dice"],
            "val_iou": val["iou"],
            "val_pixel_accuracy": val["pixel_accuracy"],
            "val_lesion_dice": val["lesion_dice"],
            "val_normal_fp": val["normal_fp"],
            "learning_rate": current_lr,
            "epoch_seconds": time.time() - epoch_start,
        })

        if val["dice"] > best_val_dice:
            best_val_dice = val["dice"]
            best_epoch = epoch + 1
            torch.save(
                {
                    "epoch": best_epoch,
                    "model_state_dict": copy.deepcopy(model.state_dict()),
                    "best_val_dice": best_val_dice,
                    "config": {
                        "condition": condition,
                        "seed": seed,
                        "image_size": IMAGE_SIZE,
                        "batch_size": BATCH_SIZE,
                        "learning_rate": LEARNING_RATE,
                        "epochs": epochs,
                    },
                },
                checkpoint_path,
            )

        pd.DataFrame(history).to_csv(history_path, index=False)  # saved every epoch

        print(
            f"[{run_name}] Epoch {epoch + 1:02d}/{epochs} "
            f"| Train {train_loss:.4f} | Val Loss {val['loss']:.4f} "
            f"| Val Dice {val['dice']:.4f} | Lesion Dice {val['lesion_dice']:.4f} "
            f"| Normal FP {val['normal_fp']}/{val['normal_total']} "
            f"| LR {current_lr:.6f} | {history[-1]['epoch_seconds']:.1f}s"
        )

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    return {
        "run": run_name,
        "condition": condition,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": round(best_val_dice, 4),
        "minutes": round((time.time() - run_start) / 60, 1),
    }


# ============================================================
# Verification (no training happens here)
# ============================================================

# 1. Architecture
n_params = sum(p.numel() for p in UNet(1, 1).parameters())
print(f"\nParameters: {n_params:,} (expected 1,926,433)")
assert n_params == 1_926_433

# 2. All five conditions load with the expected split sizes
print()
for condition, root in CONDITION_ROOTS.items():
    counts = {s: len(BUSIDataset(root, s)) for s in ["train", "validation", "test"]}
    print(f"{condition:<16} {counts}")

# 3. The new validation code reproduces the stored baseline score
ckpt = torch.load(MODEL_DIR / "baseline_unet_best.pth", map_location=DEVICE, weights_only=False)
check_model = UNet(1, 1).to(DEVICE)
check_model.load_state_dict(ckpt["model_state_dict"])
check_loader = DataLoader(
    BUSIDataset(PREPROCESSED_DIR, "validation"),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
)
check = validate_one_epoch(check_model, check_loader)

print(f"\nStored baseline validation Dice     : {ckpt['best_val_dice']:.4f}")
print(f"Recomputed with the new code        : {check['dice']:.4f}")
print(f"Lesion-only validation Dice (baseline): {check['lesion_dice']:.4f}")
print(f"Validation normal images with false positives: {check['normal_fp']} of {check['normal_total']}")

In [ ]:
smoke = run_experiment("original", seed=42, epochs=1, run_name="smoke_test")
print()
print(smoke)

smoke_history = pd.read_csv(TRAIN_RESULTS_DIR / "smoke_test_history.csv")
print()
print(smoke_history.T)

# Remove the smoke-test files so they don't mix with real results
(MODEL_DIR / "smoke_test_best.pth").unlink(missing_ok=True)
(TRAIN_RESULTS_DIR / "smoke_test_history.csv").unlink(missing_ok=True)
print("\nSmoke-test files removed.")

In [ ]:
CONDITIONS_TO_RUN = ["original", "gaussian", "median", "non_local_means"]
SEEDS = [42]


def run_is_complete(run_name, epochs=EPOCHS):
    history_path = TRAIN_RESULTS_DIR / f"{run_name}_history.csv"
    checkpoint_path = MODEL_DIR / f"{run_name}_best.pth"
    if not (history_path.exists() and checkpoint_path.exists()):
        return False
    return len(pd.read_csv(history_path)) == epochs


def summarize_run(condition, seed):
    run_name = f"{condition}_seed{seed}"
    h = pd.read_csv(TRAIN_RESULTS_DIR / f"{run_name}_history.csv")
    best = h.loc[h["val_dice"].idxmax()]
    return {
        "run": run_name,
        "condition": condition,
        "seed": seed,
        "best_epoch": int(best["epoch"]),
        "best_val_dice": round(float(best["val_dice"]), 4),
        "best_val_lesion_dice": round(float(best["val_lesion_dice"]), 4),
        "minutes": round(h["epoch_seconds"].sum() / 60, 1),
    }


summary_rows = []
total_start = time.time()

for seed in SEEDS:
    for condition in CONDITIONS_TO_RUN:
        run_name = f"{condition}_seed{seed}"

        if run_is_complete(run_name):
            print(f"Skipping {run_name}: already complete.")
        else:
            print(f"\n===== Starting {run_name} =====")
            run_experiment(condition, seed)

        summary_rows.append(summarize_run(condition, seed))
        pd.DataFrame(summary_rows).to_csv(
            TRAIN_RESULTS_DIR / "training_runs_summary.csv", index=False
        )

print(f"\nAll runs finished in {(time.time() - total_start) / 60:.1f} minutes.")
print(pd.DataFrame(summary_rows).to_string(index=False))